# Master verification (local only)

**Do not commit** `MasterVerification.ipynb` to the portfolio remote (it is gitignored). Tracked source of truth: **`MasterVerification.TEMPLATE.ipynb`**.

Copy once: `cp MasterVerification.TEMPLATE.ipynb MasterVerification.ipynb` and run locally.

**Purpose:** Re-run the PubMedQA **adaptive shootout** (vanilla vs always-RAG vs XGBoost vs DistilBERT vs LoRA routers) with fixed `n_samples=100`, `seed=42`, and the routing thresholds defined in `adaptive_rag.adaptive_shootout.run_shootout`.

**Prereqs:** GPU, `HF_TOKEN`, artifacts from `scripts/02`–`05` (CSV, FAISS, LoRA adapter; XGB joblib and DistilBERT dir optional but recommended for full table).
For **Google Colab**, use **`MasterVerification_COLAB.ipynb`** (clone + pip + same cells).


In [ ]:
# Optional: Colab / fresh env
# !pip install -r requirements.txt


In [ ]:
from pathlib import Path
import os, sys

# Repo root = parent of this file when opened as MasterVerification.ipynb at root
ROOT = Path.cwd().resolve()
if (ROOT / "src").is_dir():
    pass
elif (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT)


In [ ]:
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])


In [ ]:
import joblib
import pandas as pd
from pathlib import Path

from adaptive_rag.config import (
    ARTIFACTS_DIR,
    DISTILBERT_ROUTER_DIR,
    LORA_ROUTER_DIR,
    MEDHALLU_TRAINING_CSV,
    PUBMED_FAISS_INDEX,
    PUBMED_FAISS_MAPPING,
)
from adaptive_rag.router_baselines import train_xgboost_router

xgb_path = ARTIFACTS_DIR / "xgb_router.joblib"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

assert MEDHALLU_TRAINING_CSV.exists(), "Run scripts/02_generate_medhallu_router_csv.py first"
assert PUBMED_FAISS_INDEX.exists() and PUBMED_FAISS_MAPPING.exists(), "Run scripts/04_build_pubmed_faiss.py"
assert LORA_ROUTER_DIR.exists(), "Run scripts/03_train_lora_router.py"

if xgb_path.exists():
    xgb_model = joblib.load(xgb_path)
    print("Loaded", xgb_path)
else:
    df = pd.read_csv(MEDHALLU_TRAINING_CSV, engine="python", on_bad_lines="skip")
    xgb_model, _ = train_xgboost_router(df["prompt"].tolist(), df["label"].tolist())
    joblib.dump(xgb_model, xgb_path)
    print("Trained and saved", xgb_path)

distil_dir = DISTILBERT_ROUTER_DIR if DISTILBERT_ROUTER_DIR.exists() else None
if distil_dir is None:
    print("DistilBERT router not found; shootout will skip D_BERT")


In [ ]:
from adaptive_rag.report_benchmark import run_shootout_report

summary, df = run_shootout_report(
    faiss_index_path=PUBMED_FAISS_INDEX,
    faiss_mapping_path=PUBMED_FAISS_MAPPING,
    lora_adapter_dir=LORA_ROUTER_DIR,
    distilbert_dir=distil_dir,
    xgb_model=xgb_model,
    n_samples=100,
    seed=42,
)
from adaptive_rag.adaptive_shootout import print_summary
print_summary(summary)
df
